## RAG를 통한 LLM 답변 CODE
RAG (Retrieval-Augmented Generation)를 구현하는 과정에서
다양한 형태의 쿼리가 들어올 수 있는 점을 고려해야 합니다.

이를 위해 자연어와 JSON 쿼리를 모두 처리하고,
필요한 경우 여러 쿼리 항목을 개별적으로 처리하여 최종적으로 LLM을 통해 사용자에게 전달하는 코드를 작성해 보겠습니다.

주요 단계
- 쿼리 처리: 자연어 또는 JSON 형식의 쿼리를 처리합니다.
- 키워드 추출: 쿼리에서 주요 키워드를 추출합니다.
- Chroma DB에서 검색: 추출한 키워드를 기반으로 Chroma DB에서 관련 문서를 검색합니다.
- LLM 응답 생성: 검색된 문서들을 바탕으로 LLM이 사용자에게 적절한 응답을 생성합니다.

In [ ]:
import random
from langchain.chains import LLMChain
from langchain.agents import Tool, initialize_agent
from langchain.llms import OpenAI
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.tools import Tool
from chromadb import Client as ChromaClient
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
import re
import os
load_dotenv()
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.tools.retriever import create_retriever_tool
from langchain.chains import RetrievalQA
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# # Product Chroma DB 초기화
# database = Chroma(
#     collection_name='product_list',
#     embedding_function=OpenAIEmbeddings(),
#     persist_directory='../datas/chroma_product_list_v1_0830'
# )

In [ ]:
# LLM 및 Chroma DB 클라이언트 초기화
llm = OpenAI(model_name="gpt-4o-mini")
database = Chroma(
    collection_name='product_list',
    embedding_function=OpenAIEmbeddings(),
    persist_directory='../datas/chroma_product_list_v1_0831'
)


c:\Users\prodk\anaconda3\envs\llm\Lib\site-packages\langchain_community\llms\openai.py:254: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain_community.chat_models import ChatOpenAI`
  warnings.warn(
c:\Users\prodk\anaconda3\envs\llm\Lib\site-packages\langchain_community\llms\openai.py:1073: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain_community.chat_models import ChatOpenAI`
  warnings.warn(


In [ ]:
# retriever 객체 초기화
retriever = database.as_retriever()

In [ ]:
# LLM을 이용한 키워드 추출 함수
def extract_keywords_from_text(query):
    # LLM에게 키워드를 추출하도록 프롬프트를 생성
    prompt_template = "다음 문장에서 핵심 키워드를 추출해줘: '{query}'"
    prompt = PromptTemplate(input_variables=["query"], template=prompt_template)

    llm = OpenAI()
    # LLM에 프롬프트를 전달하여 키워드 추출
    chain = prompt | llm | StrOutputParser()

    response = chain.invoke(query)
    # 키워드를 쉼표로 구분한 결과를 리스트로 변환
    keywords = [kw.strip() for kw in response.split(",")]
    return keywords

In [ ]:
keywords = extract_keywords_from_text(query)

In [ ]:
print(keywords)

['모공관리', '제품', '추천']


In [ ]:
# Chroma DB에서 검색하는 함수
def search_chroma_db(keywords, retriever):
    # 키워드들을 조합하여 검색 수행
    query = " ".join(keywords)  # 키워드들을 공백으로 구분하여 하나의 쿼리로 결합
    results = retriever.get_relevant_documents(query)
    return results

In [ ]:
documents = search_chroma_db(keywords=keywords, retriever=retriever)

In [ ]:
documents

[Document(metadata={'Summary': "이 리뷰는 아이 소이 제품에 대한 것으로, '모공관리'에 중점을 두고 있습니다. 리뷰어는 제품 사용 후 모공 관리 효과에 대해 언급하며, 뉴트럴한 효과와 발림성을 긍정적으로 평가했습니다. 하지만 제품의 성능에 대한 확실한 변화를 느끼지 못했으며, 자극이 간혹 느껴질 수 있다는 점과 향기에 대한 불만도 보였습니다. 전반적으로, 사용 경험은 긍정적이지만, 효과에 대한 확실한 결실은 느끼지 못한 것으로 요약됩니다.\n\n**감정 분류**: 긍정적 (하나의 한계점도 포함됨)  \n**주요 초점**: 모공관리  \n**포함된 키워드**: 향기, 자극, 효과", 'brand': '아이소이', 'name': '아이소이포어타이트닝컨트롤세럼', 'price': '22500.0', 'skin_type': '모든 피부용 (민감피부도 사용가능)', 'url': 'https://www.oliveyoung.co.kr/store/goods/getGoodsDetail.do?goodsNo=A000000202978&dispCatNo=100000100010014&trackingCd=Cat100000100010014_Small&t_page=%EC%B9%B4%ED%85%8C%EA%B3%A0%EB%A6%AC%EA%B4%80&t_click=%EC%97%90%EC%84%BC%EC%8A%A4/%EC%84%B8%EB%9F%BC/%EC%95%B0%ED%94%8C_%EC%A0%84%EC%B2%B4__%EC%83%81%ED%92%88%EC%83%81%EC%84%B8&t_number=1'}, page_content="### Summary\n이 리뷰는 아이 소이 제품에 대한 것으로, '모공관리'에 중점을 두고 있습니다. 리뷰어는 제품 사용 후 모공 관리 효과에 대해 언급하며, 뉴트럴한 효과와 발림성을 긍정적으로 평가했습니다. 하지만 제품의 성능에 대한 확실한 변화를 느끼지 못했으며, 자극이 간혹 느껴질 수 있다는 점과 향기에 대한 불만도 보였습니다. 전반적으

In [ ]:
documents[0].metadata['name']

'아이소이포어타이트닝컨트롤세럼'

In [ ]:
# 응답을 생성하는 함수
def generate_response(documents, llm):
    # 각 문서에서 필요한 정보를 추출하여 요약문 생성
    responses = []

    for doc in documents[:3]:  # 상위 3개의 문서만 사용
        context = ""
        context += f"제품명: {doc.metadata['name']}\n"
        context += f"브랜드: {doc.metadata['brand']}\n"
        context += f"가격: {doc.metadata['price']}\n"
        context += f"구매링크: {doc.metadata['url']}\n"
        context += f"리뷰요약: {doc.metadata['Summary']}\n"
        context += "\n"

    # LLM에게 주어진 문맥을 바탕으로 응답을 생성하도록 요청
        # dictionary = {
        #     '제품명':',
        #     '브랜드':',
        #     '가격':,
        #     '구매링크':,
        #     '리뷰요약':documents[0].metadata['Summary'],
        # }
    # LLM에게 주어진 문맥을 바탕으로 응답을 생성하도록 요청
        prompt_template = """
        다음의 제품 정보를 바탕으로 각 제품에 대한 추천 내용을 만들어 주세요:
        {context}

        결과물(제품정보) 반드시 다음과 같이 출력이 되어야 합니다.
        예시.1)
        제품명: 메디큐브제로모공원데이크림
        브랜드: 메디큐브
        가격: 39900원
        구매링크: https://www.oliveyoung.co.kr/store/G.do?goodsNo=A000000206212
        리뷰요약: 이 제품은 모공 관리를 위한 최고의 선택입니다. 피부에 부담 없이 사용할 수 있으며, 하루 종일 촉촉함을 유지해줍니다.

        예시.2)
        제품명: AHC하이드라B5수딩토너
        브랜드: AHC
        가격: 25000원
        구매링크: https://www.oliveyoung.co.kr/store/G.do?goodsNo=A000000203568
        리뷰요약: 이 토너는 피부를 진정시키고 수분을 공급하는 데 탁월합니다. 모든 피부 타입에 적합하며 특히 건성 피부에 추천합니다.

        만일, 위의 형태로 답변이 나오지 않는 결과물은 메시지를 보여주지 마세요.
        """
        # 프롬프트 옵션들
        # 다음의 형태로 제품 정보를 최종 생성해주세요.
        # {dictionary}


        prompt = PromptTemplate(input_variables=[context], template=prompt_template)

        llm = OpenAI()
        # LLM에 프롬프트를 전달하여 키워드 추출
        chain = prompt | llm | StrOutputParser()

        response = chain.invoke(query)
        # 특수기호 제거 및 불필요한 공백 처리
        cleaned_response = re.sub(r'\s+', ' ', response).strip()  # 여러 공백을 하나로 줄이고 앞뒤 공백 제거
        cleaned_response = re.sub(r'[*_]', '', cleaned_response)  # 불필요한 특수기호 제거

        responses.append(cleaned_response)
    # 모든 제품에 대한 추천 문장을 결합하여 최종 응답 생성
    final_response = "\n\n".join(responses)
    return final_response

In [ ]:
response = generate_response(documents=documents, llm=llm)

In [ ]:
print(response)

제품명: 더페이스샵 클렌징오일 브랜드: 더페이스샵 가격: 16500원 구매링크: https://www.oliveyoung.co.kr/store/G.do?goodsNo=A000000151292 리뷰요약: 이 클렌징 오일은 모공을 깔끔하게 관리하고 수분 감도 유지하기 위한 최고의 제품입니다. 그리고 여러분의 피부를 매끈하고 건강하게 가꿔줍니다.

제품명: 이니스프리 그린티 씨드 세럼 브랜드: 이니스프리 가격: 19500원 구매링크: https://www.oliveyoung.co.kr/store/G.do?goodsNo=A000000158602 리뷰요약: 이 세럼은 피부를 촉촉하게 해주는 것은 물론이고, 모공 수렴에도 탁월한 효과가 있습니다. 촉촉한 느낌을 선호하는 분들에게 추천합니다.

제품명: 더페이스샵 제주 화산송이 모공 폼클렌저 브랜드: 더페이스샵 가격: 13000원 구매링크: https://www.oliveyoung.co.kr/store/G.do?goodsNo=A000000139276 리뷰요약: 이 클렌저는 모공을 깨끗하게 만들어주는 신세계를 보여줍니다. 피부에 부담 없이 사용할 수 있으며, 피부 결이 깨끗해지는 느낌을 줍니다.


In [ ]:
# 사용자가 질문을 하는 부분 (예시)
query = "모공관리 제품을 추천해주세요"  # 자연어 예시

In [ ]:
# 쿼리 처리 및 제품 추천
keywords = extract_keywords_from_text(query)

documents = search_chroma_db(keywords=keywords, retriever=retriever)

response = generate_response(documents=documents, llm=llm)

# 결과 출력
print(response)

제품명: 히말라야 레몬핑크솔트샤워젤 브랜드: 히말라야 가격: 7000원 구매링크: https://www.oliveyoung.co.kr/store/G.do?goodsNo=A000000211480 리뷰요약: 이 샤워젤은 부드럽고 향기로운 사용감을 제공합니다. 피부의 보습을 도와주며 상쾌함을 선사해줍니다.

제품명: 메디큐브제로모공원데이크림 브랜드: 메디큐브 가격: 39900원 구매링크: https://www.oliveyoung.co.kr/store/G.do?goodsNo=A000000206212 리뷰요약: 이 제품은 모공 관리를 위한 최고의 선택입니다. 피부에 부담 없이 사용할 수 있으며, 하루 종일 촉촉함을 유지해줍니다.

제품명: 닥터방기원 타임리버스 스킨큐어 포어 클리어 앰플 브랜드: 닥터방기원 가격: 35000원 구매링크: https://www.oliveyoung.co.kr/store/G.do?goodsNo=A000000211465 리뷰요약: 이 앰플은 모공을 깊이 관리하여 블랙헤드와 여드름을 예방해줍니다. 또한, 피부를 진정시켜주고 촉촉하게 가꿔줍니다. 피부에 부담 없이 사용할 수 있으며, 피부 타입에 상관없이 사용 가능합니다.
